In [ ]:
# 1. Configuración
from google.colab import drive
drive.mount('/content/drive')

!pip install -U wfdb plotly kaleido pywavelets imbalanced-learn tensorflow gputil psutil openpyxl -q > /dev/null

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import pywt
import cv2
import tensorflow as tf
from scipy.signal import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, BatchNormalization,
                                     Dropout, Dense, Reshape, LSTM, Bidirectional,
                                     MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D,
                                     GlobalAveragePooling2D, Concatenate, Add, Conv1D, MaxPooling1D, Activation)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.utils import to_categorical, plot_model
from tqdm.notebook import tqdm
from sklearn.utils.class_weight import compute_class_weight

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

BASE_DIR = '/content/drive/MyDrive/tesisv2'
DIRS = {k: os.path.join(BASE_DIR, v) for k, v in [
    ('models', 'modelos'), ('data', 'data'),
    ('results', 'resultados'), ('figures', 'figuras'),
    ('raw', 'mit-bih-data')
]}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

sys.path.append(f"{BASE_DIR}/scripts")
try:
    from script_monitoreo_eficiencia import MonitorEficienciaCallback
    from script_configuracion_modelo import ConfiguracionModelo
    from script_evaluacion_desempeno import EvaluacionDesempeno
except ImportError:
    print("Scripts no encontrados en 'scripts/'. Requeridos antes de continuar.")

WINDOW_SIZE = 260
CWT_SIZE = 128
LABELS_MAP = {0: "Normal", 1: "Supraventricular", 2: "Ventricular", 3: "Fusion"}
AAMI_MAP = {'N':0, 'L':0, 'R':0, 'e':0, 'j':0, 'A':1, 'a':1, 'J':1, 'S':1, 'V':2, 'E':2, 'F':3}


In [ ]:
# 2. Procesamiento
# Descargar DBs si no existen
for db in ['mitdb', 'svdb']:
    if not os.path.exists(os.path.join(DIRS['raw'], db)):
        wfdb.dl_database(db, dl_dir=os.path.join(DIRS['raw'], db))

def get_rr_features(r_peaks, idx, fs):
    pre = (r_peaks[idx] - r_peaks[idx-1])/fs if idx > 0 else 0.8
    post = (r_peaks[idx+1] - r_peaks[idx])/fs if idx < len(r_peaks)-1 else 0.8
    avg = np.mean(np.diff(r_peaks[max(0, idx-10):idx+1]))/fs if idx > 1 else 0.8
    return [pre, post, pre/(avg+1e-5), post/(avg+1e-5)]

def process_db(db_name, records):
    X, y, RR = [], [], []
    path = os.path.join(DIRS['raw'], db_name)
    for rec in tqdm(records, desc=db_name):
        try:
            r = wfdb.rdrecord(f"{path}/{rec}"); a = wfdb.rdann(f"{path}/{rec}", 'atr')
            sig = (r.p_signal[:,0] - np.mean(r.p_signal[:,0]))/(np.std(r.p_signal[:,0])+1e-8)
            for i, (loc, sym) in enumerate(zip(a.sample, a.symbol)):
                if sym in AAMI_MAP:
                    start, end = loc - 130, loc + 130
                    if start >= 0 and end <= len(sig):
                        seg = sig[start:end]
                        if len(seg) == 260:
                            if r.fs != 360: seg = resample(seg, 260)
                            X.append(seg); y.append(AAMI_MAP[sym])
                            RR.append(get_rr_features(a.sample, i, r.fs))
        except: pass
    return np.array(X), np.array(y), np.array(RR)


mit_recs = [f.replace('.hea','') for f in os.listdir(os.path.join(DIRS['raw'], 'mitdb')) if f.endswith('.hea')]
svdb_recs = [f.replace('.hea','') for f in os.listdir(os.path.join(DIRS['raw'], 'svdb')) if f.endswith('.hea')]

X_mit, y_mit, RR_mit = process_db('mitdb', mit_recs)
X_sv, y_sv, RR_sv = process_db('svdb', svdb_recs)


In [ ]:
# 3. Dataset híbrido

def augment_signal(signal, max_shift=15):
    """Jitter temporal, escalado de amplitud y ruido gaussiano."""
    shift = np.random.randint(-max_shift, max_shift + 1)
    signal_aug = np.roll(signal, shift)
    scale = np.random.uniform(0.9, 1.1)
    signal_aug = signal_aug * scale
    noise = np.random.normal(0, 0.02, len(signal_aug))
    signal_aug = signal_aug + noise
    return signal_aug

def create_dataset():
    X, y, RR = [], [], []

    idx_n = np.where(y_mit == 0)[0]
    X_n_orig = X_mit[idx_n]
    RR_n_orig = RR_mit[idx_n]

    X.extend(X_n_orig)
    y.extend(y_mit[idx_n])
    RR.extend(RR_n_orig)

    num_normal_to_augment = int(len(X_n_orig) * 0.5)
    for _ in tqdm(range(num_normal_to_augment), desc="Augmenting Normals"):
        i = np.random.randint(0, len(X_n_orig))
        s_aug = augment_signal(X_n_orig[i])
        rr_aug = RR_n_orig[i] * np.random.uniform(0.98, 1.02)
        X.append(s_aug)
        y.append(0)
        RR.append(rr_aug)

    # S + V: MIT-BIH + SVDB
    mask_sv = np.isin(y_sv, [1, 2])
    X_src = np.concatenate([X_mit, X_sv[mask_sv]])
    y_src = np.concatenate([y_mit, y_sv[mask_sv]])
    RR_src = np.concatenate([RR_mit, RR_sv[mask_sv]])

    for c in [1, 2]: # Clases S y V
        idx = np.where(y_src == c)[0]
        X.extend(X_src[idx])
        y.extend(y_src[idx])
        RR.extend(RR_src[idx])

    # F — objetivo 8000 muestras
    mask_f_mit = (y_mit == 3)
    mask_f_sv = (y_sv == 3)
    X_f = np.concatenate([X_mit[mask_f_mit], X_sv[mask_f_sv]])
    RR_f = np.concatenate([RR_mit[mask_f_mit], RR_sv[mask_f_sv]])

    X.extend(X_f)
    y.extend([3] * len(X_f))
    RR.extend(RR_f)

    needed = 8000 - len(X_f)
    if len(X_f) > 0 and needed > 0:
        for _ in tqdm(range(needed), desc="Augmenting Fusions"):
            i = np.random.randint(0, len(X_f))
            s_aug = augment_signal(X_f[i])
            rr_aug = RR_f[i] * np.random.uniform(0.95, 1.05)
            X.append(s_aug)
            y.append(3)
            RR.append(rr_aug)

    return np.array(X), np.array(y), np.array(RR)

X_all, y_all, RR_all = create_dataset()

# Split 80/20
X_train, X_test, y_train, y_test, RR_train, RR_test = train_test_split(
    X_all, y_all, RR_all, test_size=0.2, random_state=42, stratify=y_all
)

unique, counts = np.unique(y_train, return_counts=True)
for i, count in enumerate(counts):
    print(f"Clase {LABELS_MAP[i]}: {count} muestras")


In [ ]:
# 4. CWT
def gen_cwt(sig):
    imgs = []
    for s in tqdm(sig, desc="CWT"):
        c, _ = pywt.cwt(s, np.arange(1, 65), 'cmor1.5-1.0')
        s = cv2.resize(np.abs(c), (128, 128))
        imgs.append((s - s.min())/(s.max()-s.min()+1e-8))
    return np.expand_dims(np.array(imgs, dtype='float32'), -1)

X_train_img = gen_cwt(X_train)
X_test_img = gen_cwt(X_test)

X_train_1d = np.expand_dims(X_train, -1)
X_test_1d = np.expand_dims(X_test, -1)
y_train_enc = to_categorical(y_train, 4)
y_test_enc = to_categorical(y_test, 4)


In [ ]:
# 5. Modelo
def res_block(x, filters):
    if x.shape[-1] != filters:
        sc = Conv2D(filters, 1, strides=2)(x)
        stride = 2
    else:
        sc = x
        stride = 1
    y = Conv2D(filters, 3, padding='same', strides=stride)(x)
    y = BatchNormalization()(y); y = Activation('relu')(y)
    y = Conv2D(filters, 3, padding='same')(y)
    y = BatchNormalization()(y)
    x = Add()([sc, y]); x = Activation('relu')(x)
    return x

def build_model():
    # Rama 1: Imagen (ResNet)
    img_in = Input((128,128,1), name='cwt_image')
    x = Conv2D(64, 7, strides=2, padding='same')(img_in)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = MaxPooling2D(3, strides=2, padding='same')(x)
    x = res_block(x, 64); x = res_block(x, 128); x = res_block(x, 256)
    x = GlobalAveragePooling2D()(x)

    # Rama 2: Señal 1D (BiLSTM + Atención)
    sig_in = Input((260,1), name='signal_1d')
    y = Conv1D(32, 5, activation='relu', padding='same')(sig_in)
    y = MaxPooling1D(2)(y)
    y = Bidirectional(LSTM(64, return_sequences=True))(y)
    att = MultiHeadAttention(4, 64)(y, y)
    y = GlobalAveragePooling1D()(LayerNormalization()(y + att))

    # Rama 3: RR
    rr_in = Input((4,), name='rr_intervals')
    z = Dense(32, activation='relu')(rr_in)

    # Fusión
    concat = Concatenate()([x, y, z])
    out = Dense(256, activation='relu')(concat)
    out = Dropout(0.5)(out)
    out = Dense(4, activation='softmax')(out)

    return Model([img_in, sig_in, rr_in], out, name="Ultra_Model_SOTA")

model = build_model()
model.compile(optimizer=AdamW(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# 6. Entrenamiento
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights = dict(enumerate(cw))
weights[1] *= 1.2 # Boost a S
weights[3] *= 1.5 # Boost a F

ckpt = ModelCheckpoint(f"{DIRS['models']}/ultra_best.keras", save_best_only=True, monitor='val_accuracy')
callbacks = [ckpt, ReduceLROnPlateau(patience=3, factor=0.5), EarlyStopping(patience=10)]

if 'MonitorEficienciaCallback' in globals():
    monitor = MonitorEficienciaCallback('Ultra_SOTA_Training', DIRS['results'])
    callbacks.append(monitor)

history = model.fit(
    {'cwt_image': X_train_img, 'signal_1d': X_train_1d, 'rr_intervals': RR_train}, y_train_enc,
    validation_data=({'cwt_image': X_test_img, 'signal_1d': X_test_1d, 'rr_intervals': RR_test}, y_test_enc),
    epochs=40, batch_size=64, class_weight=weights,
    callbacks=callbacks
)


In [ ]:
# 7. Evaluación
model.load_weights(f"{DIRS['models']}/ultra_best.keras")

probs = model.predict({'cwt_image': X_test_img, 'signal_1d': X_test_1d, 'rr_intervals': RR_test}, batch_size=64, verbose=1)
y_pred = np.argmax(probs, axis=1)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=list(LABELS_MAP.values()), digits=4))

# --- DIMENSIÓN A: CONFIGURACIÓN ---
try:
    ConfiguracionModelo(model, DIRS['results']).run()
except Exception as e: print(f"Error Dim A: {e}")

# --- DIMENSIÓN B: EFICIENCIA ---
try:
    if 'monitor' in globals():
        monitor.imprimir_resumen()
        monitor.generar_graficos()
except Exception as e: print(f"Error Dim B: {e}")

# --- DIMENSIÓN C: DESEMPEÑO ---
try:
    eval_runner = EvaluacionDesempeno(
        y_true=y_test,
        y_pred=y_pred,
        nombre_experimento='Ultra_SOTA_Final',
        output_dir=DIRS['results']
    )
    eval_runner.run()
except Exception as e: print(f"Error Dim C: {e}")

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Greens',
            xticklabels=list(LABELS_MAP.values()), yticklabels=list(LABELS_MAP.values()))
plt.title(f"Matriz Final (Accuracy: {acc*100:.2f}%)")
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.savefig(f"{DIRS['figures']}/matriz_final.png")
plt.show()

df = pd.concat([pd.DataFrame(X_test), pd.DataFrame(RR_test), pd.DataFrame(y_test)], axis=1)
df.to_csv(os.path.join(DIRS['results'], 'mitbih_test_multimodal.csv'), header=False, index=False)


In [ ]:
# Figuras

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})
COLORS = ['#3498db', '#e74c3c', '#2ecc71', '#f1c40f']

# Ejemplos por clase
def plot_beat_examples_thesis():
    fig, axes = plt.subplots(4, 2, figsize=(15, 12), gridspec_kw={'width_ratios': [1, 2]})
    fig.suptitle('Señales ECG Representativas por Clase de Arritmia (AAMI EC57:2012)\nMIT-BIH Arrhythmia Database | Frecuencia: 360 Hz | Ventana: 260 muestras', fontsize=16, fontweight='bold', y=0.95)

    classes = [0, 1, 2, 3]
    for i, cls in enumerate(classes):
        indices = np.where(y_test == cls)[0]
        if len(indices) > 10:
             idx = indices[10] # Índice 10 para evitar outliers iniciales
        else:
             idx = indices[0]

        sig = X_test_1d[idx].flatten()
        axes[i, 1].plot(sig, color='#0B5394', linewidth=2)
        axes[i, 1].set_title(f'Clase: {LABELS_MAP[cls]}', fontweight='bold', loc='left')
        axes[i, 1].set_ylabel('Amplitud (mV)')
        axes[i, 1].grid(True, linestyle='--', alpha=0.4)
        r_peak = np.argmax(np.abs(sig))
        axes[i, 1].axvline(r_peak, color='red', linestyle='--', alpha=0.5, label='Pico R')
        if i == 0: axes[i, 1].legend()

        cwt = X_test_img[idx].squeeze()
        im = axes[i, 0].imshow(cwt, cmap='jet', aspect='auto')
        axes[i, 0].set_ylabel('Escala')
        axes[i, 0].set_xticks([])
        if i == 3: axes[i, 0].set_xlabel('Tiempo')

    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    plt.savefig(f"{DIRS['figures']}/fig1_ejemplos_latidos.png", dpi=300, bbox_inches='tight')
    plt.show()

# Distribución y pesos de clase
def plot_data_strategy():
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 6))
    fig.suptitle('Visualización de la Estrategia de Balanceo Híbrida', fontsize=20, fontweight='bold', y=1.05)

    unique, counts = np.unique(y_train, return_counts=True)
    ax1.bar(list(LABELS_MAP.values()), counts, color='#5DADE2', edgecolor='black')
    ax1.set_title('(a) Distribución Train (Post-Augmentation Moderada)', fontweight='bold')
    ax1.set_ylabel('Número de Muestras')
    for i, v in enumerate(counts): ax1.text(i, v+500, f"{v:,}", ha='center', fontweight='bold')

    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights = dict(enumerate(cw))
    weights_list = [weights[i] for i in range(4)]

    ax2.bar(list(LABELS_MAP.values()), weights_list, color='#EC7063', edgecolor='black')
    ax2.set_title('(b) Pesos de Clase Calculados (Para Penalización)', fontweight='bold')
    ax2.set_ylabel('Peso de Clase (Penalización)')
    ax2.axhline(1.0, color='green', linestyle='--', linewidth=3, label='Peso Base = 1.0')
    ax2.legend()
    for i, v in enumerate(weights_list): ax2.text(i, v+0.05, f"{v:.2f}", ha='center', fontweight='bold')

    unique_test, counts_test = np.unique(y_test, return_counts=True)
    width = 0.35
    x = np.arange(4)
    ax3.bar(x - width/2, counts, width, label='Train (Aumentado)', color='#5DADE2', edgecolor='black')
    ax3.bar(x + width/2, counts_test, width, label='Test (Original)', color='#58D68D', edgecolor='black')
    ax3.set_title('(c) Distribución Train (Aumentado) vs Test (Original)', fontweight='bold')
    ax3.set_yscale('log')
    ax3.set_ylabel('Número de Muestras (Escala Logarítmica)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(list(LABELS_MAP.values()), rotation=15)
    ax3.legend()

    plt.tight_layout()
    plt.savefig(f"{DIRS['figures']}/fig2_estrategia_datos.png", dpi=300, bbox_inches='tight')
    plt.show()

# Mapas de atención
def plot_attention_maps():
    try:
        layer_name = [l.name for l in model.layers if 'attention' in l.name][0]
        grad_model = Model(inputs=model.inputs, outputs=model.get_layer(layer_name).output)

        fig, axes = plt.subplots(4, 3, figsize=(18, 16))
        fig.suptitle('Mapas de Atención del Modelo Híbrido - Por Clase', fontsize=18, fontweight='bold', y=0.92)

        for cls in range(4):
            indices = np.where((y_test == cls) & (y_pred == cls))[0]
            if len(indices) == 0: continue
            idx = indices[0]

            inputs = {
                'cwt_image': np.expand_dims(X_test_img[idx], 0),
                'signal_1d': np.expand_dims(X_test_1d[idx], 0),
                'rr_intervals': np.expand_dims(RR_test[idx], 0)
            }

            att_output = grad_model.predict(inputs, verbose=0) # Shape (1, Timesteps, Features)
            att_map = np.mean(att_output[0], axis=-1)

            axes[cls, 0].imshow(X_test_img[idx].squeeze(), cmap='jet', aspect='auto')
            axes[cls, 0].set_title(f'{LABELS_MAP[cls]} - Escalograma CWT')
            axes[cls, 0].set_ylabel('Escala'); axes[cls, 0].set_xlabel('Tiempo')

            axes[cls, 1].plot(X_test_1d[idx], color='#2C3E50')
            axes[cls, 1].set_title(f'{LABELS_MAP[cls]} - Señal ECG Original')
            axes[cls, 1].set_ylabel('Amplitud (mV)'); axes[cls, 1].set_xlabel('Tiempo (ms)')

            # Interpolación: LSTM (variable timesteps) -> señal (260 muestras)
            att_resized = np.interp(np.linspace(0, len(att_map), 260), np.arange(len(att_map)), att_map)

            axes[cls, 2].plot(att_resized, color='red', linewidth=3)
            axes[cls, 2].fill_between(range(len(att_resized)), att_resized, color='red', alpha=0.3)
            axes[cls, 2].set_title(f'{LABELS_MAP[cls]} - Pesos de Atención')
            axes[cls, 2].set_ylabel('Peso de Atención'); axes[cls, 2].set_xlabel('Timestep')

            prob = probs[idx][cls]
            axes[cls, 2].text(0.5, 0.9, f"Pred: {LABELS_MAP[cls]}\n({prob*100:.1f}%)",
                              transform=axes[cls, 2].transAxes, ha='center',
                              bbox=dict(boxstyle="round", alpha=0.5, color='orange'))

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(f"{DIRS['figures']}/fig3_mapas_atencion.png", dpi=300)
        plt.show()
    except Exception as e:
        print(f"No se pudo generar mapa de atención (nombre de capa no encontrado): {e}")

# Matriz de confusión
def plot_final_confusion_matrix_thesis():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle('Matriz de Confusión del Modelo Híbrido CNN-LSTM-Atención', fontsize=20, fontweight='bold')

    cm = confusion_matrix(y_test, y_pred)

    # Absoluta
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
                xticklabels=list(LABELS_MAP.values()), yticklabels=list(LABELS_MAP.values()),
                annot_kws={"size": 13})
    ax1.set_title('(a) Valores Absolutos', fontsize=16)
    ax1.set_ylabel('Etiqueta Real', fontsize=14); ax1.set_xlabel('Predicción', fontsize=14)

    # Normalizada
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=ax2,
                xticklabels=list(LABELS_MAP.values()), yticklabels=list(LABELS_MAP.values()),
                annot_kws={"size": 13})
    ax2.set_title('(b) Normalizada (por fila)', fontsize=16)
    ax2.set_ylabel('Etiqueta Real', fontsize=14); ax2.set_xlabel('Predicción', fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"{DIRS['figures']}/fig4_matriz_confusion_final.png", dpi=300)
    plt.show()

plot_beat_examples_thesis()
plot_data_strategy()
plot_attention_maps()
plot_final_confusion_matrix_thesis()

if not os.path.exists(f"{DIRS['figures']}/arquitectura_modelo.png"):
    try:
        plot_model(model, to_file=f"{DIRS['figures']}/arquitectura_modelo.png",
                   show_shapes=True, show_layer_names=False, dpi=96)
    except: print("No se pudo generar diagrama (falta graphviz).")

# Diagrama conceptual
from matplotlib.patches import FancyBboxPatch, ArrowStyle

def plot_conceptual_architecture():
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    def draw_box(x, y, width, height, text, color='#EAECEE'):
        rect = FancyBboxPatch((x, y), width, height,
                              boxstyle="round,pad=0.02",
                              ec="black", fc=color, lw=1.5)
        ax.add_patch(rect)
        ax.text(x + width/2, y + height/2, text, ha='center', va='center',
                fontsize=10, fontweight='bold', wrap=True)

    def draw_arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="-|>", lw=2, color='black'))

    ax.text(0.5, 0.98, "Arquitectura Híbrida CNN-BiLSTM-Atención Multimodal", ha='center', fontsize=16, fontweight='bold')

    # Rama 1: CWT
    draw_box(0.05, 0.8, 0.25, 0.1, "Escalograma CWT\n(128x128)", color='#AED6F1')
    draw_arrow(0.175, 0.8, 0.175, 0.75)
    draw_box(0.05, 0.65, 0.25, 0.1, "CNN ResNet Profunda\n(Extracción Morfológica)", color='#F9E79F')
    draw_arrow(0.175, 0.65, 0.175, 0.6)
    draw_box(0.05, 0.5, 0.25, 0.1, "Global Avg Pooling", color='#F9E79F')

    # Rama 2: Señal 1D
    draw_box(0.375, 0.8, 0.25, 0.1, "Señal ECG 1D\n(260 muestras)", color='#AED6F1')
    draw_arrow(0.5, 0.8, 0.5, 0.75)
    draw_box(0.375, 0.65, 0.25, 0.1, "Bi-LSTM Bidireccional\n(Secuencia Temporal)", color='#ABEBC6')
    draw_arrow(0.5, 0.65, 0.5, 0.6)
    draw_box(0.375, 0.5, 0.25, 0.1, "Mecanismo de Atención\n(Multi-Head Attention)", color='#D7BDE2')

    # Rama 3: RR
    draw_box(0.7, 0.8, 0.25, 0.1, "Intervalos RR\n(4 Features)", color='#AED6F1')
    draw_arrow(0.825, 0.8, 0.825, 0.75)
    draw_box(0.7, 0.65, 0.25, 0.1, "Red Densa (MLP)\n(Procesamiento Clínico)", color='#F5B7B1')

    # Fusión
    ax.plot([0.175, 0.175, 0.45], [0.5, 0.45, 0.45], color='black', lw=2)
    draw_arrow(0.5, 0.5, 0.5, 0.4)
    ax.plot([0.825, 0.825, 0.55], [0.65, 0.45, 0.45], color='black', lw=2)

    draw_box(0.35, 0.3, 0.3, 0.1, "Capa de Concatenación\n(Fusión de Características)", color='#D5D8DC')
    draw_arrow(0.5, 0.3, 0.5, 0.25)
    draw_box(0.35, 0.15, 0.3, 0.1, "Clasificador Softmax\n(4 Clases: N, S, V, F)", color='#85C1E9')

    plt.tight_layout()
    plt.savefig(f"{DIRS['figures']}/fig5_arquitectura_conceptual.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_conceptual_architecture()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

def plot_profesor_style_architecture():
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    def draw_box(x, y, w, h, text, color, title=None):
        rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02", ec="black", fc=color, lw=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=10, fontweight='bold', wrap=True)
        if title:
            ax.text(x + w/2, y + h + 0.01, title, ha='center', va='bottom', fontsize=11, fontweight='bold', color='darkblue')

    def draw_group(x, y, w, h, label):
        rect = Rectangle((x, y), w, h, ec="gray", fc="none", lw=1, linestyle="--")
        ax.add_patch(rect)
        ax.text(x + 0.01, y + h - 0.02, label, ha='left', va='top', fontsize=12, fontweight='bold', color='gray')

    def arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", lw=2, color='black'))

    # 1. Entrada
    draw_group(0.02, 0.4, 0.12, 0.2, "1. ENTRADA")
    draw_box(0.03, 0.45, 0.1, 0.1, "Señal ECG Cruda\n(Base de Datos MIT-BIH + SVDB)", "#AED6F1")

    arrow(0.14, 0.5, 0.18, 0.5)

    # 2. Proceso
    draw_group(0.18, 0.05, 0.64, 0.9, "2. PROCESO")

    # A. Preprocesamiento
    draw_group(0.20, 0.1, 0.15, 0.8, "A. Preprocesamiento")

    draw_box(0.21, 0.75, 0.13, 0.08, "Filtrado de Ruido\n& Normalización", "#E8DAEF")
    arrow(0.275, 0.75, 0.275, 0.70)

    draw_box(0.21, 0.62, 0.13, 0.08, "Segmentación de Latidos\n(Ventana 260ms)", "#E8DAEF")
    arrow(0.275, 0.62, 0.275, 0.57)

    draw_box(0.21, 0.49, 0.13, 0.08, "Balanceo de Datos\n(Híbrido)", "#E8DAEF")
    arrow(0.275, 0.49, 0.275, 0.44)

    draw_box(0.21, 0.30, 0.13, 0.14, "Extracción de Features:\n1. Imagen CWT\n2. Señal 1D\n3. Intervalos RR", "#D2B4DE")

    arrow(0.36, 0.37, 0.40, 0.75) # A CNN
    arrow(0.36, 0.37, 0.40, 0.50) # A LSTM
    arrow(0.36, 0.37, 0.40, 0.25) # A MLP

    # B. Modelo
    draw_group(0.38, 0.1, 0.22, 0.8, "B. Modelo Híbrido (Deep Learning)")

    draw_box(0.40, 0.70, 0.18, 0.1, "CNN (ResNet)\nExtracción Visual", "#F9E79F", "Procesamiento Espacial")
    draw_box(0.40, 0.45, 0.18, 0.1, "Bi-LSTM + Atención\nSecuencia Temporal", "#ABEBC6", "Procesamiento Temporal")
    draw_box(0.40, 0.20, 0.18, 0.1, "Red Densa (MLP)\nDatos Clínicos", "#F5B7B1", "Conocimiento Experto")

    arrow(0.59, 0.75, 0.63, 0.60)
    arrow(0.59, 0.50, 0.63, 0.50)
    arrow(0.59, 0.25, 0.63, 0.40)

    # C. Clasificación
    draw_group(0.62, 0.1, 0.18, 0.8, "C. Clasificación")

    draw_box(0.64, 0.45, 0.14, 0.15, "Capa de Concatenación\n(Fusión de Características)", "#D5D8DC")
    arrow(0.71, 0.45, 0.71, 0.35)

    draw_box(0.64, 0.25, 0.14, 0.1, "Softmax\n(Probabilidades)", "#85C1E9")

    arrow(0.79, 0.30, 0.85, 0.30)

    # 3. Salida
    draw_group(0.84, 0.2, 0.14, 0.6, "3. SALIDA")

    draw_box(0.85, 0.65, 0.12, 0.1, "Clase Normal (N)", "#D5F5E3")
    draw_box(0.85, 0.50, 0.12, 0.1, "Supraventricular (S)", "#FADBD8")
    draw_box(0.85, 0.35, 0.12, 0.1, "Ventricular (V)", "#FADBD8")
    draw_box(0.85, 0.20, 0.12, 0.1, "Fusion (F)", "#FADBD8")

    ax.plot([0.85, 0.82, 0.82, 0.85], [0.30, 0.30, 0.70, 0.70], color='black', lw=1) # Bracket

    ax.text(0.5, 0.97, "Arquitectura del Sistema de Diagnóstico de Arritmias (Flujo Completo)",
            ha='center', fontsize=18, fontweight='bold', color='#17202A')

    plt.tight_layout()
    plt.savefig(f"{DIRS['figures']}/arquitectura_profesor_style.png", dpi=300)
    plt.show()

plot_profesor_style_architecture()


In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from google.colab import drive

drive.mount('/content/drive')

json_data = {
    "fecha_registro": "2025-12-03 20:50:12",
    "experimento": "Ultra_SOTA_Training",
    "indicadores": {
        "B1_tiempo_entrenamiento_segundos": 3592.78,
        "B1_tiempo_entrenamiento_minutos": 59.88
    },
    "num_epocas": 20,
    "historial_epocas": [
        {"epoca": 1, "accuracy": 0.847, "loss": 0.4316, "val_accuracy": 0.6627, "val_loss": 0.8357},
        {"epoca": 2, "accuracy": 0.9208, "loss": 0.2436, "val_accuracy": 0.9587, "val_loss": 0.1267},
        {"epoca": 3, "accuracy": 0.9403, "loss": 0.1841, "val_accuracy": 0.9339, "val_loss": 0.2052},
        {"epoca": 4, "accuracy": 0.9524, "loss": 0.1459, "val_accuracy": 0.9533, "val_loss": 0.1401},
        {"epoca": 5, "accuracy": 0.9583, "loss": 0.1276, "val_accuracy": 0.9739, "val_loss": 0.0822},
        {"epoca": 6, "accuracy": 0.9639, "loss": 0.1071, "val_accuracy": 0.9401, "val_loss": 0.1738},
        {"epoca": 7, "accuracy": 0.9669, "loss": 0.0979, "val_accuracy": 0.9646, "val_loss": 0.1069},
        {"epoca": 8, "accuracy": 0.971, "loss": 0.0853, "val_accuracy": 0.9665, "val_loss": 0.0971},
        {"epoca": 9, "accuracy": 0.9808, "loss": 0.0545, "val_accuracy": 0.9698, "val_loss": 0.0961},
        {"epoca": 10, "accuracy": 0.9837, "loss": 0.0449, "val_accuracy": 0.9851, "val_loss": 0.049},
        {"epoca": 11, "accuracy": 0.9856, "loss": 0.0407, "val_accuracy": 0.8424, "val_loss": 0.4379},
        {"epoca": 12, "accuracy": 0.9862, "loss": 0.0358, "val_accuracy": 0.9799, "val_loss": 0.0642},
        {"epoca": 13, "accuracy": 0.9875, "loss": 0.033, "val_accuracy": 0.98, "val_loss": 0.0729},
        {"epoca": 14, "accuracy": 0.9921, "loss": 0.0198, "val_accuracy": 0.9847, "val_loss": 0.0566},
        {"epoca": 15, "accuracy": 0.9931, "loss": 0.0179, "val_accuracy": 0.9832, "val_loss": 0.0636},
        {"epoca": 16, "accuracy": 0.9932, "loss": 0.0165, "val_accuracy": 0.987, "val_loss": 0.054},
        {"epoca": 17, "accuracy": 0.9957, "loss": 0.0108, "val_accuracy": 0.9875, "val_loss": 0.0547},
        {"epoca": 18, "accuracy": 0.996, "loss": 0.0094, "val_accuracy": 0.9858, "val_loss": 0.0613},
        {"epoca": 19, "accuracy": 0.9964, "loss": 0.0089, "val_accuracy": 0.9877, "val_loss": 0.0564},
        {"epoca": 20, "accuracy": 0.9973, "loss": 0.0067, "val_accuracy": 0.988, "val_loss": 0.0573}
    ]
}

def plot_training_history_from_data(data, save_path=None):
    history = data.get('historial_epocas')
    acc = [epoch['accuracy'] for epoch in history]
    val_acc = [epoch['val_accuracy'] for epoch in history]
    loss = [epoch['loss'] for epoch in history]
    val_loss = [epoch['val_loss'] for epoch in history]
    epochs_range = range(len(acc))

    plt.style.use('default')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))

    ax1.plot(epochs_range, acc, 'o-', color='#1f77b4', label='Train', markersize=6)
    ax1.plot(epochs_range, val_acc, 's-', color='#ff7f0e', label='Validation', markersize=6)
    ax1.set_title('Accuracy durante el Entrenamiento', fontsize=16, fontweight='bold')
    ax1.set_xlabel('Epoca', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.grid(True, which='both', linestyle='--', linewidth=0.5, color='#cccccc')
    ax1.legend(loc='lower right', fontsize=12)
    ax1.set_ylim(bottom=min(min(val_acc), min(acc)) - 0.05, top=1.01)
    ax1.set_facecolor('#f9f9f9')

    ax2.plot(epochs_range, loss, 'o-', color='#1f77b4', label='Train', markersize=6)
    ax2.plot(epochs_range, val_loss, 's-', color='#ff7f0e', label='Validation', markersize=6)
    ax2.set_title('Loss durante el Entrenamiento', fontsize=16, fontweight='bold')
    ax2.set_xlabel('Epoca', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.grid(True, which='both', linestyle='--', linewidth=0.5, color='#cccccc')
    ax2.legend(loc='upper right', fontsize=12)
    ax2.set_facecolor('#f9f9f9')

    for ax in [ax1, ax2]:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
        ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
        ax.set_xlim(-0.5, len(acc) - 0.5)

    fig.tight_layout(pad=3.0)

    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Guardado en: {save_path}")

    plt.show()

output_image_path = '/content/drive/MyDrive/tesisv2/figuras/historial_entrenamiento_final.png'
plot_training_history_from_data(json_data, save_path=output_image_path)
